# SegFormer-B0, their procedure, sliding vs whole-image inference

Their intended entry point is `Experiments/Main.ipynb`: build the config, process
it, construct a `PatchTrainer`, call `train()`, pickle the result. This notebook
is that same cell with the seven-model loop replaced by a two-arm loop over the
**inference mode**, for `mmseg_segformer_b0` only:

| arm | `model.slide` | what the segmentor does |
|---|---|---|
| `slide` | `True` | `SlidingWindowSegModel`, crop and stride from the checkpoint's own `test_cfg` (1024 / 768) |
| `whole` | `False` | `WrappedSegModel`, one forward over the full frame |

Nothing else changes. The augmentation, the 256 px centre patch written into the
*normalised* image, the ignore-label footprint, the sign-gradient ascent with
`clamp_(0,1)` and the correctly-classified-pixels CE objective are all theirs,
untouched.

### Read this before interpreting the training arms

**At the training resolution the two arms are the same model.** `train.height` x
`train.width` is 1024 x 1024, and `slide_windows(1024, 1024, (1024,1024),
(768,768))` yields exactly one window covering the whole frame, with
`count == 1` everywhere. So `slide=True` during *training* is bit-identical to
`slide=False` -- the two patches differ only by dataloader/CUDA nondeterminism,
not by inference mode. Both arms are trained here anyway because that is what
was asked for, and because an identical-by-construction pair is a useful control
on run-to-run variance; do not read a difference between them as an effect of
the inference mode.

**The mode bites at evaluation.** Cityscapes val is scored at the native
1024 x 2048 frame (`multi_scale=False`, `flip=False`, so `gen_sample` applies no
resize and no crop). There `slide` is **three** windows and `whole` is one, and
each slide window is an independent forward -- a patch cannot influence pixels
outside the windows that contain it. The evaluation section below therefore
scores **both patches under both modes**, a 2 x 2, which is where the real
answer to "sliding vs whole" is.


In [ ]:
import os, sys, time, pickle, json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision.utils import save_image

# Their cell 0 appended the Colab project dir to sys.path. Walk up from the
# notebook instead, so this works in a plain checkout. log_patch_address in the
# yaml is RELATIVE ("Experiments/runs/"), so the repo root also has to be the
# working directory -- hence the chdir.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs" / "config.yaml").is_file())
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CONFIG = "configs/config.yaml"

from utils.utils import get_config_from_yaml, process_config, print_config
from trainer.trainer import PatchTrainer
from dataset.cityscapes import Cityscapes
from metrics.performance import SegmentationMetric
from patch.create import Patch
from pretrained_models.models import Models
from tqdm import tqdm

MODEL = "mmseg_segformer_b0"
ARMS = (("slide", True), ("whole", False))   # (tag, model.slide)

# Knobs. Leave at None to use the yaml as-is.
END_EPOCH = None     # e.g. 1 for a timing pass; yaml says 30
CKPT_DIR  = None     # e.g. "checkpoints" if the mmseg cfg+pth pair is local
N_IMAGES  = None     # eval on the first N val frames only; None = all 500

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def build_config(slide):
    """Their get_config_from_yaml, then the overrides. config is a namedtuple,
    so every override goes through _replace rather than item assignment."""
    config = get_config_from_yaml(CONFIG, model=MODEL)
    model = config.model._replace(slide=slide)
    if CKPT_DIR is not None:
        model = model._replace(ckpt_dir=CKPT_DIR)
    config = config._replace(model=model)
    if END_EPOCH is not None:
        config = config._replace(train=config.train._replace(end_epoch=END_EPOCH))
    return config


def save_run(config, save, tag, wall):
    """Their pickle, plus the final.pt that scripts/eval_patch.py reads.

    THE `space` FIELD IS NOT DECORATION. trainer.py optimises the patch in
    NORMALISED space -- apply_patch writes it into an image input_transform has
    already normalised, so clamp_(0,1) bounds it in normalised units. Inverted,
    [0,1] normalised is roughly R [0.485,0.714], G [0.456,0.680],
    B [0.406,0.631] in true pixel space: a narrow light-grey band, never black
    and never saturated. Load this tensor into a pipeline expecting a [0,1]
    PIXEL patch and you silently evaluate a different, much darker patch.
    """
    patch, iou = save
    stem = f"{config.model.name}_{tag}"
    run_dir = Path(config.experiment.log_patch_address) / "patch" / stem
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(config.experiment.log_patch_address + stem + ".p", "wb") as fh:
        pickle.dump(save, fh)                       # their format, unchanged

    torch.save({
        "patch": patch.detach().cpu(), "space": "normalised",
        "mean": MEAN.flatten().tolist(), "std": STD.flatten().tolist(),
        "patch_size": config.patch.size, "patch_loc": config.patch.loc,
        "model_name": config.model.name, "ckpt_dir": config.model.ckpt_dir,
        "slide": config.model.slide,
        "train_crop": [config.train.height, config.train.width],
        "epochs": config.train.end_epoch, "epsilon": config.optimizer.init_lr,
        "ignore_label": config.train.ignore_label,
        "iou_trace": iou, "wall_clock_s": wall,
        "source": "shekhar et al. procedure, mmseg checkpoint",
    }, run_dir / "final.pt")

    p = patch.detach().cpu().float()
    # Two PNGs, because the raw tensor is not what the patch looks like.
    # final_patch.png is the true appearance (pixel = normalised*std + mean) --
    # use this one in figures. final_patch_normspace.png reads the bare tensor
    # as pixels, which is what their Main.ipynb cell 3 renders, so it is the
    # space their published patch images are in; it shows more contrast than
    # the patch actually has.
    save_image((p * STD + MEAN).clamp(0, 1), run_dir / "final_patch.png")
    save_image(p.clamp(0, 1), run_dir / "final_patch_normspace.png")
    return run_dir


print(f"repo   : {REPO}")
print(f"device : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print_config(build_config(True))

# Train

Their `Main.ipynb` cell 2, verbatim in its four calls and their order --

```
config      = get_config_from_yaml(..., model=<name>)
main_logger = process_config(config)
train_obj   = PatchTrainer(config, main_logger)
save        = train_obj.train()      ->  pickled to log_patch_address
```

-- with the loop running over the two inference modes instead of over the seven
models. Each arm writes `Experiments/runs/mmseg_segformer_b0_<arm>.p` (their
pickle) and `Experiments/runs/patch/mmseg_segformer_b0_<arm>/final.pt`.

One epoch is a full pass over the 2975-image train list at `batch_size 1`;
measured at roughly 360 s/epoch for b0, so the yaml's 30 epochs is about 3 h
**per arm**. Set `END_EPOCH = 1` in the cell above for a timing pass first.

In [ ]:
runs = {}
for tag, slide in ARMS:
    config = build_config(slide)
    main_logger = process_config(config)
    main_logger.info(f"[arm  ] {tag}: model.slide={slide}")
    main_logger.info(f"[patch] size={config.patch.size} loc={config.patch.loc} "
                     f"epsilon={config.optimizer.init_lr}")
    main_logger.info(f"[train] crop={config.train.height}x{config.train.width} "
                     f"epochs={config.train.end_epoch} "
                     f"batch={config.train.batch_size}")

    t0 = time.time()
    train_obj = PatchTrainer(config, main_logger)     # prints the resolved
    save = train_obj.train()                          # [infer] line
    wall = time.time() - t0

    run_dir = save_run(config, save, tag, wall)
    runs[tag] = {"config": config, "patch": save[0].detach().cpu(),
                 "iou": save[1], "wall": wall, "dir": run_dir}
    print(f"[{tag}] {wall/60:.1f} min -> {run_dir}")

# The patches

Their cell 3 is `plt.imshow` on the bare patch tensor. That is the *normalised*
space the optimiser works in, not the patch's appearance -- the left column
below is their render, the right column is the same patch converted to pixels.
The right column is the honest one, and it is a narrow light-grey band because
`clamp_(0,1)` bounds the patch in normalised units.

In [ ]:
fig, axes = plt.subplots(len(runs), 2, figsize=(6, 3 * len(runs)), squeeze=False)
for r, (tag, run) in enumerate(runs.items()):
    p = run["patch"].float()
    axes[r][0].imshow(p.clamp(0, 1).permute(1, 2, 0))
    axes[r][0].set_title(f"{tag}: raw tensor (their cell 3)", fontsize=9)
    axes[r][1].imshow((p * STD + MEAN).clamp(0, 1).permute(1, 2, 0))
    axes[r][1].set_title(f"{tag}: true appearance", fontsize=9)
    for ax in axes[r]:
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

for tag, run in runs.items():
    p = run["patch"].float()
    px = (p * STD + MEAN)
    print(f"{tag:6s} normalised [{p.min():.3f},{p.max():.3f}]  "
          f"pixel [{px.min():.3f},{px.max():.3f}]  "
          f"= RGB [{255*px.min():.0f},{255*px.max():.0f}]")

# The control the markdown above promises: at the 1024x1024 crop both arms are
# one window, so any difference here is run-to-run noise, not the mode.
if {"slide", "whole"} <= set(runs):
    d = (runs["slide"]["patch"] - runs["whole"]["patch"]).abs()
    print(f"\n|slide - whole| : mean {d.mean():.4f} max {d.max():.4f} "
          f"({100*(d > 0).float().mean():.1f}% of pixels differ)")

# Evaluate: 2 x 2, native 1024 x 2048

Their evaluation protocol, as in `Table1and2.ipynb`: the Cityscapes val list with
`multi_scale=False, flip=False`, so frames arrive at the native 1024 x 2048 with
no resize and no crop; the patch is applied with their own `Patch.apply_patch`,
which also overwrites the label under the patch with `ignore_label`, so the patch
footprint leaves both the loss and the metric; mIoU is `SegmentationMetric`
pooled over the dataset, 19 classes, void excluded.

Two things worth stating plainly:

* **The patch is centred on the 2048-wide frame**, so at 256 px it covers 3.1 %
  of the evaluation frame (it was 6.25 % of the 1024 x 1024 training crop).
  Their published number is a 200 x 200 patch, i.e. 1.9 % of the frame, so these
  drops are **not** directly comparable to theirs without noting the area.
* **Their clean baseline and their patched number are not scored on the same
  label.** The clean one comes from `pretrained_models/CityscapeVal.ipynb` via
  `metrics.performance.pixAcc_miou`, which applies no patch and scores the
  *full* label; the patched one comes from `Table1and2.ipynb`, which excludes
  the footprint. So their clean mIoU counts the pixels the patch will later
  cover and their patched mIoU does not. Both are reported below: `clean
  (theirs)` is the number their published drops are measured from and the one
  to quote against their GitHub, `clean (footprint ex)` is the same logits
  scored on the patched label, which makes the drop paired.

Everything else matches theirs by construction: `base_size`/`crop_size` are
dead on this path (`multi_scale=False` means `gen_sample` never calls
`multi_scale_aug`, the only caller of `rand_crop`), `shuffle` and `batch_size`
do not matter because `SegmentationMetric` pools `total_inter`/`total_union`
over the pass, and `Patch.apply_patch`'s `loc="center"` is the same
`(w - size)//2, (h - size)//2` their notebook hardcodes.

Each inference mode builds its model once and scores the clean frame plus every
patch in one pass. On a partial run (`N_IMAGES` set) classes absent from the
accumulated ground truth contribute 0 to `IoU.mean()` instead of being skipped,
which drags mIoU down -- quote full-set numbers only.

In [ ]:
results = {}
for emode, slide in ARMS:
    cfg = build_config(slide)
    dev = cfg.experiment.device

    model = Models(cfg)
    model.get()                      # prints [ckpt] and [infer] for this mode

    ds = Cityscapes(root=cfg.dataset.root, list_path=cfg.dataset.val,
                    num_classes=cfg.dataset.num_classes,
                    multi_scale=False, flip=False,
                    ignore_label=cfg.train.ignore_label,
                    base_size=cfg.test.base_size,
                    crop_size=(cfg.test.height, cfg.test.width))
    dl = torch.utils.data.DataLoader(dataset=ds, batch_size=1, shuffle=False,
                                     num_workers=cfg.test.num_workers,
                                     pin_memory=cfg.test.pin_memory,
                                     drop_last=False)
    apply_patch = Patch(cfg).apply_patch

    # m_clean is scored on the UNPATCHED label -- their clean baseline, which
    # comes from a different notebook than their patched number and counts the
    # pixels the patch will later cover. m_clean_f is the same logits scored on
    # the patched label, which is not theirs but makes the drop paired. Both
    # are reported; see the markdown above.
    m_clean = SegmentationMetric(cfg); m_clean.reset()
    m_clean_f = SegmentationMetric(cfg); m_clean_f.reset()
    m_adv = {}
    for ptag in runs:
        m_adv[ptag] = SegmentationMetric(cfg); m_adv[ptag].reset()

    n = 0
    with torch.no_grad():
        for batch in tqdm(dl, desc=f"eval {emode}", unit="img"):
            image, label = batch[0].to(dev), batch[1].to(dev)

            # The footprint is loc/size-determined, so any arm's patch gives
            # the same ref_label; it is only used for the paired baseline.
            _, ref_label = apply_patch(
                image, label, runs[next(iter(runs))]["patch"].to(dev))
            out_clean = model.predict(image, ref_label.shape)
            m_clean.update(out_clean, label)          # theirs
            m_clean_f.update(out_clean, ref_label)    # paired

            for ptag, run in runs.items():
                patched_image, patched_label = apply_patch(
                    image, label, run["patch"].to(dev))
                m_adv[ptag].update(
                    model.predict(patched_image, patched_label.shape),
                    patched_label)

            n += 1
            if N_IMAGES is not None and n >= N_IMAGES:
                break

    pix_c, miou_c = m_clean.get()
    pix_f, miou_f = m_clean_f.get()
    results[emode] = {
        "frames": n, "frames_available": len(ds),
        "partial": N_IMAGES is not None and n < len(ds),
        "inference": emode, "slide": slide,
        # whole-image is their GitHub procedure, so only that column is
        # comparable to their published numbers.
        "inference_is_theirs": emode == "whole",
        "eval_resolution": [cfg.test.height, cfg.test.width],
        "patch_size": cfg.patch.size, "patch_loc": cfg.patch.loc,
        "clean_protocol": "unpatched label (theirs, CityscapeVal.ipynb)",
        "clean": {"pixAcc": float(pix_c), "mIoU": float(miou_c),
                  "per_class_iou": [float(v) for v in m_clean.get(full=True)]},
        "clean_footprint_excluded": {
            "pixAcc": float(pix_f), "mIoU": float(miou_f),
            "per_class_iou": [float(v) for v in m_clean_f.get(full=True)]},
        "patched": {},
    }
    for ptag, m in m_adv.items():
        pix_a, miou_a = m.get()
        results[emode]["patched"][ptag] = {
            "pixAcc": float(pix_a), "mIoU": float(miou_a),
            "drop_mIoU": float(miou_c - miou_a),
            "drop_mIoU_paired": float(miou_f - miou_a),
            "per_class_iou": [float(v) for v in m.get(full=True)]}

    del model
    torch.cuda.empty_cache()

In [ ]:
hdr = f"{'patch trained':<22}" + "".join(f"{'eval ' + e:>22}" for e, _ in ARMS)
print(hdr); print("-" * len(hdr))
print(f"{'clean (theirs)':<22}" + "".join(
    f"{results[e]['clean']['mIoU']:>22.4f}" for e, _ in ARMS))
print(f"{'clean (footprint ex)':<22}" + "".join(
    f"{results[e]['clean_footprint_excluded']['mIoU']:>22.4f}"
    for e, _ in ARMS))
print("-" * len(hdr))
for ptag in runs:
    row = f"{ptag:<22}"
    for e, _ in ARMS:
        r = results[e]["patched"][ptag]
        row += f"{r['mIoU']:>13.4f}" + f"(-{r['drop_mIoU']:.4f})".rjust(9)
    print(row)
print("\ndrop is against their clean baseline (unpatched label).")
print("the 'eval whole' column is their GitHub procedure -- the only one "
      "comparable\nto their published numbers; 'eval slide' is the "
      "checkpoint's own test_cfg.")

part = [e for e in results if results[e]["partial"]]
if part:
    print(f"\n[warn] partial set ({results[part[0]]['frames']} of "
          f"{results[part[0]]['frames_available']} frames) -- absent classes "
          f"drag mIoU down; do not quote these.")

base = build_config(True)
out = Path("Experiments/runs/patch") / f"{MODEL}_slide_vs_whole.json"
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, "w") as fh:
    json.dump({"model_name": MODEL,
               "ckpt_dir": base.model.ckpt_dir,
               "train_crop": [base.train.height, base.train.width],
               "trained_epochs": {t: int(r["config"].train.end_epoch)
                                  for t, r in runs.items()},
               "wall_clock_s": {t: r["wall"] for t, r in runs.items()},
               "results": results}, fh, indent=2)
print(f"\n-> {out}")

### Per-class IoU

19-class train-id order. The two column pairs are the same patch under the two
inference modes, so a class that survives under `slide` and collapses under
`whole` is a class whose pixels sit outside the windows containing the patch.

In [ ]:
names = ["road","sidewalk","building","wall","fence","pole","traffic light",
         "traffic sign","vegetation","terrain","sky","person","rider","car",
         "truck","bus","train","motorcycle","bicycle"]

for ptag in runs:
    print(f"\n=== patch trained with {ptag} ===")
    print(f"{'class':<15}" + "".join(f"{'clean -> ' + e:>19}" for e, _ in ARMS))
    for c, nm in enumerate(names):
        row = f"{nm:<15}"
        for e, _ in ARMS:
            cl = results[e]["clean"]["per_class_iou"][c]
            ad = results[e]["patched"][ptag]["per_class_iou"][c]
            row += f"{cl:>9.3f} ->{ad:>8.3f}"
        print(row)